# Component 2 - Feature Engineering

Runs `scripts/build_features.py` on the ingestion outputs (run *Player Stats Ingestion* first) and shows the results. The script holds the logic.

Each row is one player in one game, with features known before tip-off for every stat we get prop lines for (points, rebounds, assists, threes). Rows are built from the player's and teams' state after their most recent earlier game, so no row sees its own result.

- **Train:** past games, with actual stats as targets.
- **Upcoming:** games in the look-ahead window, same features, no targets.

Teammates out is the one exception to the before-tip-off rule for past games: it comes from the box score (who didn't play), while upcoming games use the injury report (who is listed Out).

No modelling happens here.

In [ ]:
LEAGUE = "nba"

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "build_features.py").exists())
sys.path.insert(0, str(root / "scripts"))

import importlib
import fetch_player_stats, build_features as bf
importlib.reload(fetch_player_stats); importlib.reload(bf)  # pick up script edits without restarting the kernel

import pandas as pd
pd.set_option("display.max_columns", None)

## Build

In [ ]:
train, upcoming, feats = bf.build(LEAGUE)

## Features by group

In [ ]:
groups = {
    "Player form (rolling, per-36, season avg, spread)": [c for c in feats if c.split("_")[0] in bf.target_stats()],
    "Minutes, role and usage": [c for c in feats if c.startswith(("min_", "starter_", "usage_"))],
    "Shot and rebound volume, shooting %": [c for c in feats if c.split("_")[0] in bf.VOLUME or "_pct_" in c],
    "Schedule, experience and position": [c for c in feats if c in ("home", "days_rest", "back_to_back", "prior_games", "prior_games_season") or c.startswith("pos_")],
    "Team and opponent (incl. vs league average and by position)": [c for c in feats if c.startswith(("team_", "opp_"))],
    "Teammates out": [c for c in feats if c.startswith("out_")],
}
for name, cols in groups.items():
    print(f"{name} ({len(cols)}): " + ", ".join(cols))
    print()
assert sum(map(len, groups.values())) == len(feats), set(feats) - {c for cols in groups.values() for c in cols}

## Correlation with each target

Rows with 10+ prior games. A quick check that the features carry signal, not a model.

In [ ]:
d = train[train["prior_games"] >= 10]
corr = pd.DataFrame({t: d[feats].astype(float).corrwith(d[t]) for t in bf.target_stats()})
corr.sort_values("pts", ascending=False).round(3)

## Upcoming rows

In [ ]:
upcoming[upcoming["eligible"]].head(20)

## Save

In [ ]:
bf.save(LEAGUE, train, upcoming)